In [0]:
source_path = '/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115913_928637_0.json', name='fraud_watchlist_20261117_115913_928637_0.json', size=365, modificationTime=1794916755000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115919_423282_1.json', name='fraud_watchlist_20261117_115919_423282_1.json', size=379, modificationTime=1794916760000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115924_820734_2.json', name='fraud_watchlist_20261117_115924_820734_2.json', size=380, modificationTime=1794916766000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115930_264592_3.json', name='fraud_watchlist_20261117_115930_264592_3.json', size=383, modificationTime=1794916771000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115935_666263_4.json', na

In [0]:
input_stream = (spark.readStream
  .format('cloudFiles')
  .option('cloudFiles.format', 'json')
  .option('cloudFiles.schemaLocation', '/Volumes/finguard/source/fraud_watchlist/schema/')
  .option('cloudfiles.inferSchema', 'true')
  .load(source_path)
)


In [0]:
from pyspark.sql import functions as F
transform_df= input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingest_timestamp")
  )


In [0]:
streaming_query = (
  transform_df
  .writeStream
  .format('delta')
  .outputMode('append')
  .option('checkpointLocation', '/Volumes/finguard/source/fraud_watchlist/checkpoint/')
  .trigger(availableNow=True)
  .toTable('finguard.bronze.fraud_watchlist_batch_test')
)


In [0]:
%sql

select * from finguard.silver.fraud_watchlist


watchlist_id,watch_type,entity_id,risk_level,action,reason_code,reason_description,status,effective_from,reported_by,reported_source,city,country,source_file,bronze_ingestion_timestamp,silver_ingestion_timestamp
WL000006,CARD,4009109391119268,HIGH,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:12:00.000Z,SOC,Manual,Mumbai,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115941_125825_5.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
WL000005,CARD,4260650191792026,CRITICAL,ADD,ACCOUNT_TAKEOVER,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:07:00.000Z,Customer Care,CRM,Bengaluru,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115935_666263_4.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
WL000004,CARD,4717327980595032,HIGH,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:04:00.000Z,AML Team,External Feed,Chennai,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115930_264592_3.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
WL000003,CARD,5329839542961842,LOW,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:02:00.000Z,AML Team,External Feed,Delhi,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115924_820734_2.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
WL000002,CARD,4413337312552355,HIGH,ADD,BLACKLISTED_MERCHANT,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:01:00.000Z,SOC,External Feed,Pune,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115919_423282_1.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
WL000001,CARD,5008514036965665,HIGH,ADD,PHISHING,Auto-generated fraud watchlist entry,ACTIVE,2026-06-18T09:00:00.000Z,SOC,Manual,Hyderabad,India,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20261117_115913_928637_0.json,2026-11-17T12:35:19.627Z,2026-11-17T12:53:33.038Z
